<div style="background-color: #ffffff; color: #000000; padding: 30px;">
<img src="../media/images/kisz_logo.png" width="192" height="69" align="right" style="margin-right: 50px; margin-bottom:50px;"> 
<h1> Time Series Analysis and Forecasting
</div>



<div style="background-color: #f6a800; color: #ffffff; padding: 10px;">
<h2> Part A: Foundations & Data Exploration
<h2> Notebook 3. Handling missing data
</div>

Missing data is common in real-world time series, caused by sensor failures, reporting gaps, holidays, or errors. Handling missing values correctly is crucial because they can distort trends, seasonality, and model forecasts.

We are going to use as example the UCI dataset for a single household power consumption. You can find more information about it [here](../data/datasets.md).



In [ ]:
# packages import
import pandas as pd
import numpy as np
import missingno as msno
import matplotlib.pyplot as plt
import seaborn as sns

import nb_config

from src.plotting import plot_imputed_data

# data loading
df = pd.read_csv(nb_config.HOUSEHOLD_POWER_PATH, sep=';', decimal='.',
                 low_memory=False)

# Set the style for the plots
sns.set_theme(style="whitegrid")

<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3>1. Preparing the data
</div>

First, need to prepare the dataset.

In [ ]:
# force numerical data type for columns 2 to 7
cols_to_numeric = df.columns[2:8]
for col in cols_to_numeric:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Combine Date and Time into a datetime column
df['Datetime'] = pd.to_datetime(df['Date'] + ' ' + df['Time'], 
                                format='%d/%m/%Y %H:%M:%S', errors='coerce')

# Set as index
df.set_index('Datetime', inplace=True)

# Drop the original Date and Time columns
df.drop(['Date', 'Time'], axis=1, inplace=True)

We want to be sure that our index is in full range for the sampling frequency (1 min).

In [ ]:
# Generate full datetime index for the entire range
full_index = pd.date_range(start=df.index.min(), end=df.index.max(), freq='1min')

# Check for missing timestamps in the index
missing_timestamps = full_index.difference(df.index)
print("Number of missing timestamps:", len(missing_timestamps))
if len(missing_timestamps) != 0:
    print(missing_timestamps)

Let's take a look to the time series itself!

In [ ]:
plt.figure(figsize=(12,5))

# Create a cyclical palette (e.g., hsv) for 12 months
colors = sns.color_palette("cividis_r", 12)

# Plot using 'hour' vs 'Voltage', hue='month', with cyclical colors
sns.lineplot(x=df.index.hour, y=df['Voltage'], hue=df.index.month_name(), palette=colors, alpha=0.8)

# X-axis ticks every 3 hours, remove padding
plt.xticks(ticks=[0,6,12,18,23])
plt.xlim(0, 23)

plt.xlabel('Hour of Day')
plt.ylabel('Voltage')
plt.title('Daily Pattern of Voltage by Month')

# Center the legend on top
plt.legend(title='Month', loc='lower left', fontsize=8, title_fontsize=10, ncol=3)

plt.show()

In [ ]:
# Create day of week and hour columns
df['day_of_week'] = df.index.dayofweek  # 0=Monday, 6=Sunday
df['hour'] = df.index.hour

# Pivot table: average voltage for each hour and day
weekly_pattern = df.pivot_table(index='hour', columns='day_of_week', values='Voltage', aggfunc='mean')

plt.figure(figsize=(12,6))
sns.heatmap(weekly_pattern, cmap='viridis', annot=False)
plt.xlabel('Day of Week (0=Mon)')
plt.ylabel('Hour of Day')
plt.title('Weekly Pattern of Voltage')
plt.show()



In [ ]:
df.drop(['day_of_week', 'hour'], axis=1, inplace=True)

<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3>2. Detecting missing values
</div>

Now we can check whether the time series has missing data and how much.

In [ ]:
# show absolute values of missing data
df.isna().sum()

In [ ]:
# show percentage of missing data
missing_percent = df.isna().mean() * 100
missing_percent

In [ ]:
# Visualize missing data across the entire DataFrame
msno.matrix(df, freq="2MS");

It looks like the missing values are consistent, and when a value is missing for a feature it will also be missing for all the rest. Let's check it.

In [ ]:
# This returns True if missing values are aligned across all columns
all_missing_aligned = df.isna().all(axis=1) | df.notna().all(axis=1)
print(all_missing_aligned.all())

We were right. We will try to find out now, in which positions are the missing values and how big are the gaps.

In [ ]:
# Boolean Series: True where any column is missing
is_missing = df.isna().any(axis=1)

# Identify consecutive segments
group_id = (is_missing != is_missing.shift()).cumsum()

# Length of each segment
gap_lengths = is_missing.groupby(group_id).sum()

# Type of each segment: True if missing
gap_types = is_missing.groupby(group_id).first()

# Keep only missing segments
missing_gaps = gap_lengths[gap_types]

# Start timestamp of each missing segment
gap_starts = is_missing.groupby(group_id).apply(lambda x: x.index[0])[gap_types]

# Combine start + length
missing_info = pd.DataFrame({
    'start_timestamp': gap_starts.values,
    'gap_length': missing_gaps.values
}).set_index('start_timestamp').T

# pd.set_option('display.max_columns', None)
missing_info  # just output the DataFrame

We find in most of the cases gaps of just one value, in several cases gaps with gaps in the order of the two digits (30 to 80) and in a very few bigger gaps (800 to over 7000 timestamp gaps).

We will just leave aside for pedagogical reasons a more rigurous analysis to determine causes and consequences of the missing values on our dataset and just test different imputation techniques with these different cases to see how do they perform.

<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3>3. Imputing with mean and median
</div>

We have selected for study and comparison three gaps of sizes 2, 43 and 891 timestamps respectively. We will impute and compare with the values for the day before.

In [ ]:
# declare the dates for the gaps we want to study
gap_dates_list = ["2006-12-30 10:08:00", "2008-10-25 10:28:00", "2009-08-13 05:00:00"]
gap_dates = pd.to_datetime(gap_dates_list, format="%Y-%m-%d %H:%M:%S")

# declare the durations of the gaps in minutes
gap_durations = [2, 43, 891]

# packs the dates and duration in a dictionary
gaps = dict(zip(gap_dates, gap_durations))

In [ ]:
# define the imputation function
def mean_median_imputation(series):
    imputed_mean = series.fillna(series.mean()).copy()
    imputed_median = series.fillna(series.median()).copy()
    return {'Mean':imputed_mean, 'Median':imputed_median}


# plot the imputed data
fig, axes = plot_imputed_data(df, gaps, mean_median_imputation)

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()
